<a href="https://colab.research.google.com/github/jaysacheti8/antproj/blob/main/dagster_ml_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [21]:
!pip install dagster dagster-webserver pandas numpy matplotlib seaborn scikit-learn


In [22]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

from dagster import asset, Definitions


In [23]:
@asset
def raw_data():
    df = pd.read_csv("/content/heart.csv")
    return df


In [59]:
@asset
def eda(raw_data):
    import matplotlib.pyplot as plt
    import seaborn as sns
    import os

    df = raw_data.copy()

    plot_dir = "plots"          # visible in Colab
    os.makedirs(plot_dir, exist_ok=True)   # 🔥 THIS WAS MISSING

    # Target Distribution
    plt.figure(figsize=(6,4))
    ax = sns.countplot(x="target", data=df)
    for c in ax.containers:
        ax.bar_label(c)

    plt.title("Target Distribution")
    plt.tight_layout()
    plt.savefig(f"{plot_dir}/target_distribution.png")
    plt.close()

    # Correlation Heatmap
    plt.figure(figsize=(10,8))
    sns.heatmap(
        df.select_dtypes(include="number").corr(),
        annot=True,
        cmap="coolwarm",
        fmt=".2f"
    )
    plt.title("Correlation Heatmap")
    plt.tight_layout()
    plt.savefig(f"{plot_dir}/correlation_heatmap.png")
    plt.close()

    return df


In [62]:
from dagster import asset
import pandas as pd

@asset
def raw_data():
    return pd.read_csv("/content/heart.csv")


In [63]:
from dagster import materialize

materialize([raw_data, eda])


2026-01-31 07:51:57 +0000 - dagster - DEBUG - __ephemeral_asset_job__ - 533cd2ed-4e3c-4a3d-9705-0b77dfa4f091 - 658 - RUN_START - Started execution of run for "__ephemeral_asset_job__".
2026-01-31 07:51:57 +0000 - dagster - DEBUG - __ephemeral_asset_job__ - 533cd2ed-4e3c-4a3d-9705-0b77dfa4f091 - 658 - ENGINE_EVENT - Executing steps in process (pid: 658)
2026-01-31 07:51:57 +0000 - dagster - DEBUG - __ephemeral_asset_job__ - 533cd2ed-4e3c-4a3d-9705-0b77dfa4f091 - 658 - RESOURCE_INIT_STARTED - Starting initialization of resources [io_manager].
2026-01-31 07:51:57 +0000 - dagster - DEBUG - __ephemeral_asset_job__ - 533cd2ed-4e3c-4a3d-9705-0b77dfa4f091 - 658 - RESOURCE_INIT_SUCCESS - Finished initialization of resources [io_manager].
2026-01-31 07:51:57 +0000 - dagster - DEBUG - __ephemeral_asset_job__ - 533cd2ed-4e3c-4a3d-9705-0b77dfa4f091 - 658 - LOGS_CAPTURED - Started capturing logs in process (pid: 658).
2026-01-31 07:51:57 +0000 - dagster - DEBUG - __ephemeral_asset_job__ - 533cd2ed-4

In [60]:
import os

print("Current working directory:", os.getcwd())
print("Files here:", os.listdir())


Current working directory: /content
Files here: ['.config', '__pycache__', 'dagster_pipeline.py', 'heart.csv', '.ipynb_checkpoints', 'sample_data']


In [57]:
@asset
def split_data(eda):
    X = eda.drop("target", axis=1)
    y = eda["target"]

    return train_test_split(
        X, y,
        test_size=0.2,
        random_state=42
    )


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [26]:
@asset
def decision_tree(split_data):
    X_train, X_test, y_train, y_test = split_data
    model = DecisionTreeClassifier()
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    return accuracy_score(y_test, preds)


In [27]:
@asset
def random_forest(split_data):
    X_train, X_test, y_train, y_test = split_data
    model = RandomForestClassifier()
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    return accuracy_score(y_test, preds)


In [28]:
@asset
def logistic_regression(split_data):
    X_train, X_test, y_train, y_test = split_data
    model = LogisticRegression(max_iter=1000)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    return accuracy_score(y_test, preds)


In [29]:
@asset
def knn(split_data):
    X_train, X_test, y_train, y_test = split_data
    model = KNeighborsClassifier()
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    return accuracy_score(y_test, preds)


In [30]:
defs = Definitions(
    assets=[
        raw_data,
        eda,
        split_data,
        decision_tree,
        random_forest,
        logistic_regression,
        knn
    ]
)


In [31]:
!dagster dev


/usr/local/lib/python3.12/dist-packages/click/core.py:824: SupersessionWarning: Function `dev_command` is superseded and its usage is discouraged. Use 'dg dev' instead.
  return callback(*args, **kwargs)
2026-01-31 07:32:10 +0000 - dagster - INFO - Using temporary directory /content/.tmp_dagster_home_y211183k for storage. This will be removed when dagster dev exits.
2026-01-31 07:32:10 +0000 - dagster - INFO - To persist information across sessions, set the environment variable DAGSTER_HOME to a directory to use.
2026-01-31 07:32:12 +0000 - dagster - INFO - Launching Dagster services...
Usage: dagster dev [OPTIONS]
Try 'dagster dev --help' for help.

Error: No arguments given and no [tool.dagster] block in pyproject.toml found.


In [32]:
%%writefile dagster_pipeline.py
from dagster import asset, Definitions
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score


@asset
def raw_data():
    return pd.read_csv("/content/heart.csv")


@asset
def eda(raw_data):
    plt.figure()
    sns.countplot(x="target", data=raw_data)
    plt.show()

    plt.figure(figsize=(10,8))
    sns.heatmap(raw_data.corr(), cmap="coolwarm")
    plt.show()

    return raw_data


@asset
def split_data(eda):
    X = eda.drop("target", axis=1)
    y = eda["target"]
    return train_test_split(X, y, test_size=0.2, random_state=42)


@asset
def decision_tree(split_data):
    X_train, X_test, y_train, y_test = split_data
    model = DecisionTreeClassifier()
    model.fit(X_train, y_train)
    return accuracy_score(y_test, model.predict(X_test))


@asset
def random_forest(split_data):
    X_train, X_test, y_train, y_test = split_data
    model = RandomForestClassifier()
    model.fit(X_train, y_train)
    return accuracy_score(y_test, model.predict(X_test))


@asset
def logistic_regression(split_data):
    X_train, X_test, y_train, y_test = split_data
    model = LogisticRegression(max_iter=1000)
    model.fit(X_train, y_train)
    return accuracy_score(y_test, model.predict(X_test))


@asset
def knn(split_data):
    X_train, X_test, y_train, y_test = split_data
    model = KNeighborsClassifier()
    model.fit(X_train, y_train)
    return accuracy_score(y_test, model.predict(X_test))


defs = Definitions(
    assets=[
        raw_data,
        eda,
        split_data,
        decision_tree,
        random_forest,
        logistic_regression,
        knn,
    ]
)


Overwriting dagster_pipeline.py


In [33]:
!dagster dev -f dagster_pipeline.py


/usr/local/lib/python3.12/dist-packages/click/core.py:824: SupersessionWarning: Function `dev_command` is superseded and its usage is discouraged. Use 'dg dev' instead.
  return callback(*args, **kwargs)
2026-01-31 07:32:15 +0000 - dagster - INFO - Using temporary directory /content/.tmp_dagster_home_cfnjchp7 for storage. This will be removed when dagster dev exits.
2026-01-31 07:32:15 +0000 - dagster - INFO - To persist information across sessions, set the environment variable DAGSTER_HOME to a directory to use.
2026-01-31 07:32:17 +0000 - dagster - INFO - Launching Dagster services...
2026-01-31 07:32:28 +0000 - dagster.daemon - INFO - Instance is configured with the following daemons: ['AssetDaemon', 'BackfillDaemon', 'FreshnessDaemon', 'QueuedRunCoordinatorDaemon', 'SchedulerDaemon', 'SensorDaemon']
2026-01-31 07:32:30 +0000 - dagster-webserver - INFO - Serving dagster-webserver on http://127.0.0.1:3000 in process 73386
2026-01-31 07:32:32 +0000 - dagster.daemon - INFO - Received i

In [34]:
!dagster dev -f dagster_pipeline.py --host 0.0.0.0 --port 3000


/usr/local/lib/python3.12/dist-packages/click/core.py:824: SupersessionWarning: Function `dev_command` is superseded and its usage is discouraged. Use 'dg dev' instead.
  return callback(*args, **kwargs)
2026-01-31 07:32:39 +0000 - dagster - INFO - Using temporary directory /content/.tmp_dagster_home_wbqvgkjj for storage. This will be removed when dagster dev exits.
2026-01-31 07:32:39 +0000 - dagster - INFO - To persist information across sessions, set the environment variable DAGSTER_HOME to a directory to use.
2026-01-31 07:32:42 +0000 - dagster - INFO - Launching Dagster services...
2026-01-31 07:32:56 +0000 - dagster.daemon - INFO - Instance is configured with the following daemons: ['AssetDaemon', 'BackfillDaemon', 'FreshnessDaemon', 'QueuedRunCoordinatorDaemon', 'SchedulerDaemon', 'SensorDaemon']
2026-01-31 07:32:58 +0000 - dagster-webserver - INFO - Serving dagster-webserver on http://0.0.0.0:3000 in process 73705
2026-01-31 07:33:01 +0000 - dagster - INFO - KeyboardInterrupt r

In [35]:
!pip install pyngrok


In [ ]:
!dagster dev -f dagster_pipeline.py --host 0.0.0.0 --port 3000


/usr/local/lib/python3.12/dist-packages/click/core.py:824: SupersessionWarning: Function `dev_command` is superseded and its usage is discouraged. Use 'dg dev' instead.
  return callback(*args, **kwargs)
2026-01-31 07:53:21 +0000 - dagster - INFO - Using temporary directory /content/.tmp_dagster_home_37yrop9h for storage. This will be removed when dagster dev exits.
2026-01-31 07:53:21 +0000 - dagster - INFO - To persist information across sessions, set the environment variable DAGSTER_HOME to a directory to use.
2026-01-31 07:53:23 +0000 - dagster - INFO - Launching Dagster services...


2026-01-31 07:53:36 +0000 - dagster.daemon - INFO - Instance is configured with the following daemons: ['AssetDaemon', 'BackfillDaemon', 'FreshnessDaemon', 'QueuedRunCoordinatorDaemon', 'SchedulerDaemon', 'SensorDaemon']
2026-01-31 07:53:37 +0000 - dagster-webserver - INFO - Serving dagster-webserver on http://0.0.0.0:3000 in process 84254
2026-01-31 07:54:01 +0000 - dagster.daemon.QueuedRunCoordinatorDaemon - INFO - Priority sorting and checking tag concurrency limits for queued runs.
2026-01-31 07:54:04 +0000 - dagster.daemon.QueuedRunCoordinatorDaemon - INFO - Launched 1 runs.
2026-01-31 07:54:08 +0000 - dagster - DEBUG - __ASSET_JOB - 73851eec-1837-4f0d-bb7c-e60715310105 - 84881 - RUN_START - Started execution of run for "__ASSET_JOB".
2026-01-31 07:54:08 +0000 - dagster - DEBUG - __ASSET_JOB - 73851eec-1837-4f0d-bb7c-e60715310105 - 84881 - ENGINE_EVENT - Executing steps using multiprocess executor: parent process (pid: 84881)
2026-01-31 07:54:08 +0000 - dagster - DEBUG - __ASSET_J

In [65]:
!ngrok config add-authtoken 38wKiRU0uvKQ0zE2Pnk02RsZDKT_8527BFzrSCUGsQaV87cmv


Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [66]:
from pyngrok import ngrok

public_url = ngrok.connect(3000)
print(public_url)


NgrokTunnel: "https://kadence-unrecruited-wintrily.ngrok-free.dev" -> "http://localhost:3000"
